# Organoid skeleton construction: step-by-step walkthrough

This notebook walks through building a **biology-aware skeleton graph** for one intestinal organoid. It follows the same spirit as `tutorial_segmentation.ipynb`: keep intermediate objects visible, make the important parameters easy to edit, and provide quick visual feedback.

The skeleton is not a generic medial axis. It is built from fresh crypt detections and uses straight graph edges only:

- `body -> neck -> tip` for a simple crypt;
- `body -> neck -> bend -> tip` when a bend node is requested;
- `body -> neck -> branch -> daughter tips` when an initial crypt candidate splits into multiple refined crypts.

The most useful tuning loop is near the end of the notebook: change the crypt detection, filtering, refinement, and bend-node parameters, rebuild the skeleton, and inspect the mesh overlay plus node/edge tables.

In [1]:
# --- Imports ---
import os
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

from organograph.io_utils.dataset_config import load_mesh_dataset_config
from organograph.io_utils.path_parsing import discover_mesh_paths, parse_mesh_path
from organograph.mesh.OrganoidMesh import OrganoidMesh
from organograph.mesh.geodesics import compute_geodesics_dijkstra

from organograph.crypts.filters import filter_crypts_by_hks_percent, filter_crypts_by_size

# During skeleton development, Jupyter may keep an older version of these modules
# in memory. Drop only the skeleton modules so imports below reflect the files on disk.
import importlib
import sys

for _mod in [
    "organograph.skeleton.build",
    "organograph.skeleton.datatypes",
    "organograph.skeleton.geometry",
    "organograph.skeleton.io",
    "organograph.skeleton.primitives",
    "organograph.skeleton",
]:
    sys.modules.pop(_mod, None)
importlib.invalidate_caches()
from organograph.plotting.meshes import plot_organoid_mesh
from organograph.plotting.skeletons import plot_mesh_with_skeleton

from organograph.skeleton.build import (
    build_skeleton_from_crypt_detections,
    detect_crypts_for_skeleton,
)
from organograph.skeleton.geometry import (
    crypt_attachment_direction,
    crypt_bend_angle,
    crypt_path_length,
    crypt_straight_distance,
    crypt_tortuosity,
    number_of_crypts,
    number_of_split_crypts,
)


pd.set_option("display.max_colwidth", 120)

## 1) Dataset paths + organoid selection

Edit this cell first. In the usual case you only need `DATASET`, `TIMEPOINT`, `WELL`, and `ORGANOID_ID`. The notebook reads `mesh_config.json` to fill in the zarr, round, and mesh folder names. If an organoid lives somewhere unusual, set `MESH_PATH` directly.

In [ ]:
# -----------------------
# CONFIG: edit these
# -----------------------

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "notebooks" and (NOTEBOOK_DIR / "notebooks").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"
PROJECT_ROOT = NOTEBOOK_DIR.parent

DATASET = "20251201"  # e.g. "20250929" or "20251201"
TIMEPOINT = "day4p5"
WELL = "B03"          # e.g. "B03"
ORGANOID_ID = "144"   # mesh file stem, without .vtp

MESH_DATA_DIR = PROJECT_ROOT.parent / "NicoleData" / DATASET / "fractal_output"
MESH_CONFIG_PATH = PROJECT_ROOT.parent / "NicoleData" / DATASET / "mesh_config.json"
VOCAB_PATH = PROJECT_ROOT / "sim" / "vocab_with_meta.npz"

# Optional escape hatch for unusual paths. Leave as None for the normal dataset/timepoint/well/id path.
MESH_PATH = None

print("PROJECT_ROOT     =", PROJECT_ROOT)
print("MESH_DATA_DIR   =", MESH_DATA_DIR)
print("MESH_CONFIG_PATH=", MESH_CONFIG_PATH)
print("VOCAB_PATH      =", VOCAB_PATH)
print("TIMEPOINT       =", TIMEPOINT)
print("WELL            =", WELL)
print("ORGANOID_ID     =", ORGANOID_ID)

PROJECT_ROOT     = /home/fmoller/Projects/LearningOrganoids/OrganoGraph
MESH_DATA_DIR   = /home/fmoller/Projects/LearningOrganoids/NicoleData/20250929/fractal_output
MESH_CONFIG_PATH= /home/fmoller/Projects/LearningOrganoids/NicoleData/20250929/mesh_config.json
VOCAB_PATH      = /home/fmoller/Projects/LearningOrganoids/OrganoGraph/sim/vocab_with_meta.npz
TIMEPOINT       = day3p5
WELL            = B03
ORGANOID_ID     = 144


In [3]:
# --- Build the mesh path from dataset/timepoint/well/organoid id ---
mesh_cfg = load_mesh_dataset_config(str(MESH_CONFIG_PATH))

if MESH_PATH is None:
    if TIMEPOINT not in mesh_cfg["zarr_name_by_tp"]:
        raise KeyError(f"TIMEPOINT={TIMEPOINT!r} is not present in {MESH_CONFIG_PATH}")
    if not isinstance(WELL, str) or len(WELL) < 2:
        raise ValueError("WELL should look like 'B03'")

    MESH_PATH = (
        Path(MESH_DATA_DIR)
        / TIMEPOINT
        / mesh_cfg["zarr_name_by_tp"][TIMEPOINT]
        / WELL[0]
        / WELL[1:]
        / mesh_cfg["round_by_tp"][TIMEPOINT]
        / "meshes"
        / mesh_cfg["meshname_by_tp"][TIMEPOINT]
        / f"{ORGANOID_ID}.vtp"
    )

MESH_PATH = Path(MESH_PATH)
if not MESH_PATH.exists():
    # Helpful fallback: list close matches for the requested well/timepoint.
    mesh_paths = discover_mesh_paths(
        data_dir=str(MESH_DATA_DIR),
        timepoints=[TIMEPOINT],
        zarr_names=mesh_cfg["zarr_name_by_tp"],
        rounds=mesh_cfg["round_by_tp"],
        meshes=mesh_cfg["meshname_by_tp"],
        wells={TIMEPOINT: [WELL]},
    )
    print("Requested mesh path does not exist:")
    print(" ", MESH_PATH)
    print(f"Found {len(mesh_paths)} meshes for {TIMEPOINT}/{WELL}. First matches:")
    for p in mesh_paths[:10]:
        print(" ", Path(p).stem, p)
    raise FileNotFoundError("Check DATASET, TIMEPOINT, WELL, and ORGANOID_ID.")

try:
    rec = parse_mesh_path(str(MESH_PATH))
    label_uid = rec.get("label_uid", MESH_PATH.stem)
except Exception:
    rec = {}
    label_uid = MESH_PATH.stem

print("Selected mesh:", MESH_PATH)
print("label_uid    :", label_uid)
print("parsed ids   :", rec)

Requested mesh path does not exist:
  /home/fmoller/Projects/LearningOrganoids/NicoleData/20250929/fractal_output/day3p5/r0.zarr/B/03/0_fused_zillum_registered/meshes/nnorg_linked_multi_annotated_class/144.vtp
Found 100 meshes for day3p5/B03. First matches:
  10 /home/fmoller/Projects/LearningOrganoids/NicoleData/20250929/fractal_output/day3p5/r0.zarr/B/03/0_fused_zillum_registered/meshes/nnorg_linked_multi_annotated_class/10.vtp
  100 /home/fmoller/Projects/LearningOrganoids/NicoleData/20250929/fractal_output/day3p5/r0.zarr/B/03/0_fused_zillum_registered/meshes/nnorg_linked_multi_annotated_class/100.vtp
  101 /home/fmoller/Projects/LearningOrganoids/NicoleData/20250929/fractal_output/day3p5/r0.zarr/B/03/0_fused_zillum_registered/meshes/nnorg_linked_multi_annotated_class/101.vtp
  102 /home/fmoller/Projects/LearningOrganoids/NicoleData/20250929/fractal_output/day3p5/r0.zarr/B/03/0_fused_zillum_registered/meshes/nnorg_linked_multi_annotated_class/102.vtp
  103 /home/fmoller/Projects/Lea

FileNotFoundError: Check DATASET, TIMEPOINT, WELL, and ORGANOID_ID.

## 2) Load and prepare the mesh

The HKS-based crypt detector needs a Laplace-Beltrami eigendecomposition. Normalizing the mesh is optional, but it matches the usual segmentation workflow and makes lengths easier to compare while tuning.

In [ ]:
NORMALIZE_MESH = True
NORMALIZE_SCALE = 10.0
EIGEN_K = 225

mesh = OrganoidMesh(str(MESH_PATH))
mesh.label_uid = label_uid

if NORMALIZE_MESH:
    center, scale = mesh.normalize_inplace(scale=NORMALIZE_SCALE, center="mean")
    print("Normalized mesh with center=", center, "scale=", scale)

# This also builds the mass matrix used by vertex_areas().
mesh._ensure_eigendecomposition(k=EIGEN_K)

vocab = np.load(str(VOCAB_PATH), allow_pickle=True)

print("vertices:", mesh.v.shape)
print("faces   :", mesh.f.shape)
print("vocab entries:", np.asarray(vocab["vocab"]).shape)

In [ ]:
# Quick mesh sanity check.
plot_organoid_mesh(mesh, backend="plotly", alpha=0.9, show_colorbar=False)

## 3) Tuning parameters

The skeleton adapter intentionally reruns detection from the fresh HKS candidate screen. This is useful for split crypts: the initial candidate patch can serve as the skeleton trunk/stem, while refinement identifies daughter tips.

`DETECTION_KWARGS` controls HKS candidate detection, optional refinement, and neckline normalization. `FILTER_KWARGS` controls reusable crypt filters. `BUILD_KWARGS` controls graph construction, especially bend-node insertion.

In [ ]:
# --- Crypt detection parameters ---
DETECTION_KWARGS = dict(
    L_ref=None,
    crypt_vocab_idx=None,
    threshold=0.50,
    refine_crypts=True,
    refine_threshold=0.00,
    refine_only_if_area_at_least=5.0,
    min_refined_frac_of_parent=0.05,
    geodesic_kwargs=None,
    extend_max=2.0,
    disc_resolution=200,
    neck_search_interval=(0.8, 2.0),
)

# --- Candidate filters ---
FILTER_KWARGS = dict(
    use_hks_filter=True,
    min_percent_greater=4.0,
    hks_t_min=None,
    hks_t_max=10.0,
    use_size_filter=True,
    min_patch_verts=25,
    min_patch_area=5.0,
)

# --- Skeleton graph parameters ---
BUILD_KWARGS = dict(
    body_center=None,
    add_bend_nodes=False,
    bend_strategy="none",  # "none", "midpoint", "crypt_centroid_midsection"
    metadata={"label_uid": label_uid, "mesh_path": str(MESH_PATH)},
)

def make_filter_list(**kw):
    filters = []
    if kw.get("use_hks_filter", True):
        filters.append(
            lambda patches, **inner: filter_crypts_by_hks_percent(
                patches,
                min_percent_greater=kw["min_percent_greater"],
                t_min=kw.get("hks_t_min"),
                t_max=kw.get("hks_t_max"),
                **inner,
            )
        )
    if kw.get("use_size_filter", True):
        filters.append(
            lambda patches, **inner: filter_crypts_by_size(
                patches,
                min_patch_verts=kw["min_patch_verts"],
                min_patch_area=kw.get("min_patch_area"),
                **inner,
            )
        )
    return filters or None

## 4) Build one skeleton

The function below is the main tuning loop. It returns the normalized detections, the `SkeletonGraph`, and intermediate segmentation variables such as initial candidate patches and refined child patches.

In [ ]:
def run_skeleton_pipeline(
    *,
    detection_kwargs=None,
    filter_kwargs=None,
    build_kwargs=None,
    show_tables=True,
    show_plots=True,
):
    detection_kwargs = dict(DETECTION_KWARGS if detection_kwargs is None else detection_kwargs)
    filter_kwargs = dict(FILTER_KWARGS if filter_kwargs is None else filter_kwargs)
    build_kwargs = dict(BUILD_KWARGS if build_kwargs is None else build_kwargs)

    filter_fn_list = make_filter_list(**filter_kwargs)

    detections, skel_vars = detect_crypts_for_skeleton(
        mesh,
        vocab,
        geodesic_fn=compute_geodesics_dijkstra,
        filter_fn_list=filter_fn_list,
        return_intermediates=True,
        **detection_kwargs,
    )

    graph = build_skeleton_from_crypt_detections(
        vertices=mesh.v,
        faces=mesh.f,
        crypt_detections=detections,
        **build_kwargs,
    )

    measurement_rows = []
    for crypt_id in graph.crypt_ids():
        direction = crypt_attachment_direction(graph, crypt_id)
        measurement_rows.append(
            dict(
                crypt_id=crypt_id,
                path_length=crypt_path_length(graph, crypt_id),
                straight_distance=crypt_straight_distance(graph, crypt_id),
                tortuosity=crypt_tortuosity(graph, crypt_id),
                bend_angle_rad=crypt_bend_angle(graph, crypt_id),
                attachment_x=direction[0],
                attachment_y=direction[1],
                attachment_z=direction[2],
            )
        )
    measurements = pd.DataFrame(measurement_rows)

    print(f"Skeleton crypts: {number_of_crypts(graph)}")
    print(f"Split crypts   : {number_of_split_crypts(graph)}")
    print(f"Nodes / edges  : {len(graph.nodes)} / {len(graph.edges)}")

    if show_plots:
        display(
            plot_mesh_with_skeleton(
                mesh.v,
                mesh.f,
                graph,
                backend="plotly",
                mesh_alpha=0.22,
                show_node_labels=True,
            )
        )

    if show_tables:
        display(measurements)
        display(graph.to_node_dataframe())
        display(graph.to_edge_dataframe())

    return detections, graph, skel_vars, measurements


detections, graph, skel_vars, measurements = run_skeleton_pipeline()

## 5) Inspect split candidates and stems

For skeletons, split handling starts from the initial candidate patches. If refinement divides a parent patch into multiple children, the parent becomes one skeleton crypt with a branch node and daughter tips. Any parent vertices not assigned to daughters are stored as `stem_vertices` metadata in the normalized detection.

In [ ]:
summary = []
for det in detections:
    daughters = det.get("daughters", [])
    summary.append(
        dict(
            crypt_id=det.get("crypt_id"),
            n_parent_vertices=len(det.get("crypt_vertices", [])),
            n_daughters=len(daughters),
            n_stem_vertices=len(det.get("stem_vertices", [])),
            bottom_vertex_id=det.get("bottom_vertex_id"),
            L_crypt=det.get("L_crypt"),
        )
    )

pd.DataFrame(summary)

## Notes and TODOs

- Neck nodes are placed at the centroid of the estimated neckline ring when a normalized distance field is available, so they should lie inside the mesh rather than on the surface.
- Bend nodes are optional. `midpoint` is a placeholder; `crypt_centroid_midsection` uses crypt-region vertices near the middle of the neck-tip chord.
- Branch points for split crypts use explicit split coordinates if provided; otherwise the builder falls back to a stem centroid or a midpoint between neck and daughter tips.
- Future primitive fitting can attach to `node.primitive_attachment` or `edge.primitive_attachment` without changing the skeleton topology.